In [0]:
# Load all datasets

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

customers = spark.createDataFrame([
    (1, "John ", "Hyderabad"),
    (2, "Alice", "Chennai"),
    (3, None, "Bangalore"),
    (4, "Bob", "Hyderabad"),
    (5, "Charlie", "Chennai")
], ["customer_id", "name", "city"])

cars = spark.createDataFrame([
    (101, "Toyota", "Camry", 30000),
    (102, "Honda", "Civic", -20000),
    (103, "Hyundai", "i20", 15000),
    (104, "Ford", "Mustang", 45000),
    (105, "Tesla", "Model 3", None)
], ["car_id", "brand", "model", "price"])

sales = spark.createDataFrame([
    (1, 1, 101, "2024-01-15", 1),
    (2, 2, 102, "2024-01-20", 2),
    (3, 99, 103, "2024-02-10", 1),
    (4, 4, 104, "2024-02-25", 1),
    (5, 1, 103, "2024-03-05", 3),
    (6, 5, 101, "2024-03-15", 1)
], ["sale_id", "customer_id", "car_id", "sale_date", "quantity"])

dealers = spark.createDataFrame([
    (201, "Prime Motors", "Hyderabad"),
    (202, "City Auto", "Chennai"),
    (203, "Speed Cars", "Bangalore")
], ["dealer_id", "name", "city"])

sales_dealer = spark.createDataFrame([
    (1, 201),
    (2, 202),
    (4, 201),
    (5, 203),
    (6, 202)
], ["sale_id", "dealer_id"])

print("All datasets loaded successfully")

All datasets loaded successfully


In [0]:
# Check schema and counts

print("Customers schema:")
customers.printSchema()
print(f"Count: {customers.count()}")

print("\nCars schema:")
cars.printSchema()
print(f"Count: {cars.count()}")

print("\nSales schema:")
sales.printSchema()
print(f"Count: {sales.count()}")

print("\nDealers schema:")
dealers.printSchema()
print(f"Count: {dealers.count()}")

print("\nSales_Dealer schema:")
sales_dealer.printSchema()
print(f"Count: {sales_dealer.count()}")

Customers schema:
root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

Count: 5

Cars schema:
root
 |-- car_id: long (nullable = true)
 |-- brand: string (nullable = true)
 |-- model: string (nullable = true)
 |-- price: long (nullable = true)

Count: 5

Sales schema:
root
 |-- sale_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- car_id: long (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- quantity: long (nullable = true)

Count: 6

Dealers schema:
root
 |-- dealer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

Count: 3

Sales_Dealer schema:
root
 |-- sale_id: long (nullable = true)
 |-- dealer_id: long (nullable = true)

Count: 5


In [0]:
# Identify nulls in each table

print("Nulls in customers:")
customers.select([count(when(col(c).isNull(), c)).alias(c) for c in customers.columns]).show()

print("Nulls in cars:")
cars.select([count(when(col(c).isNull(), c)).alias(c) for c in cars.columns]).show()

print("Nulls in sales:")
sales.select([count(when(col(c).isNull(), c)).alias(c) for c in sales.columns]).show()

Nulls in customers:
+-----------+----+----+
|customer_id|name|city|
+-----------+----+----+
|          0|   1|   0|
+-----------+----+----+

Nulls in cars:
+------+-----+-----+-----+
|car_id|brand|model|price|
+------+-----+-----+-----+
|     0|    0|    0|    1|
+------+-----+-----+-----+

Nulls in sales:
+-------+-----------+------+---------+--------+
|sale_id|customer_id|car_id|sale_date|quantity|
+-------+-----------+------+---------+--------+
|      0|          0|     0|        0|       0|
+-------+-----------+------+---------+--------+



In [0]:
# Check for duplicates

print("Duplicates in customers:")
customers.groupBy(customers.columns).count().filter("count > 1").show()

print("Duplicates in cars:")
cars.groupBy(cars.columns).count().filter("count > 1").show()

print("Duplicates in sales:")
sales.groupBy(sales.columns).count().filter("count > 1").show()

Duplicates in customers:
+-----------+----+----+-----+
|customer_id|name|city|count|
+-----------+----+----+-----+
+-----------+----+----+-----+

Duplicates in cars:
+------+-----+-----+-----+-----+
|car_id|brand|model|price|count|
+------+-----+-----+-----+-----+
+------+-----+-----+-----+-----+

Duplicates in sales:
+-------+-----------+------+---------+--------+-----+
|sale_id|customer_id|car_id|sale_date|quantity|count|
+-------+-----------+------+---------+--------+-----+
+-------+-----------+------+---------+--------+-----+



In [0]:
# Identify invalid values

print("Cars with negative prices:")
cars.filter(col("price") < 0).show()

print("Sales with invalid customer_id:")
sales.join(customers, "customer_id", "left_anti").show()

print("Sales with invalid car_id:")
sales.join(cars, "car_id", "left_anti").show()

Cars with negative prices:
+------+-----+-----+------+
|car_id|brand|model| price|
+------+-----+-----+------+
|   102|Honda|Civic|-20000|
+------+-----+-----+------+

Sales with invalid customer_id:
+-----------+-------+------+----------+--------+
|customer_id|sale_id|car_id| sale_date|quantity|
+-----------+-------+------+----------+--------+
|         99|      3|   103|2024-02-10|       1|
+-----------+-------+------+----------+--------+

Sales with invalid car_id:
+------+-------+-----------+---------+--------+
|car_id|sale_id|customer_id|sale_date|quantity|
+------+-------+-----------+---------+--------+
+------+-------+-----------+---------+--------+



In [0]:
# Handle nulls

customers = customers.fillna({"name": "Unknown", "city": "Unknown"})
cars = cars.fillna({"price": 0})
sales = sales.fillna({"quantity": 1})

print("Nulls handled successfully")

Nulls handled successfully


In [0]:
# Fix negative prices and trim strings

cars = cars.withColumn("price", when(col("price") < 0, 0).otherwise(col("price")))
customers = customers.withColumn("name", trim(col("name"))).withColumn("city", trim(col("city")))
dealers = dealers.withColumn("name", trim(col("name"))).withColumn("city", trim(col("city")))

print("Prices fixed and strings trimmed")

Prices fixed and strings trimmed


In [0]:
# Remove invalid foreign keys

sales = sales.join(customers, "customer_id", "inner").join(cars, "car_id", "inner")
sales = sales.select("sale_id", "customer_id", "car_id", "sale_date", "quantity")

print(f"Clean sales count: {sales.count()}")

Clean sales count: 5


In [0]:
# Create validation report

total_customers = customers.count()
total_cars = cars.count()
total_sales = sales.count()
null_customer_names = customers.filter(col("name") == "Unknown").count()
null_car_prices = cars.filter(col("price") == 0).count()

validation_report = spark.createDataFrame([
    ("total_customers", str(total_customers)),
    ("total_cars", str(total_cars)),
    ("total_sales", str(total_sales)),
    ("null_customer_names", str(null_customer_names)),
    ("null_car_prices", str(null_car_prices))
], ["metric", "value"])

validation_report.show()

+-------------------+-----+
|             metric|value|
+-------------------+-----+
|    total_customers|    5|
|         total_cars|    5|
|        total_sales|    5|
|null_customer_names|    1|
|    null_car_prices|    2|
+-------------------+-----+



In [0]:
# Customer revenue

sales = sales.withColumn("sale_date", to_date(col("sale_date")))
df = sales.join(customers, "customer_id").join(cars, "car_id")
df = df.withColumn("revenue", col("price") * col("quantity"))

customer_revenue = df.groupBy("customer_id", "name").agg(sum("revenue").alias("total_revenue")).orderBy(desc("total_revenue"))

print("Customer revenue:")
customer_revenue.show()

Customer revenue:
+-----------+-------+-------------+
|customer_id|   name|total_revenue|
+-----------+-------+-------------+
|          1|   John|        75000|
|          4|    Bob|        45000|
|          5|Charlie|        30000|
|          2|  Alice|            0|
+-----------+-------+-------------+



In [0]:
# Brand-wise sales

brand_sales = df.groupBy("brand").agg(
    sum("quantity").alias("total_quantity"),
    sum("revenue").alias("total_revenue")
).orderBy(desc("total_revenue"))

print("Brand-wise sales:")
brand_sales.show()

Brand-wise sales:
+-------+--------------+-------------+
|  brand|total_quantity|total_revenue|
+-------+--------------+-------------+
| Toyota|             2|        60000|
|   Ford|             1|        45000|
|Hyundai|             3|        45000|
|  Honda|             2|            0|
+-------+--------------+-------------+



In [0]:
# City-wise revenue

city_revenue = df.groupBy("city").agg(sum("revenue").alias("total_revenue")).orderBy(desc("total_revenue"))

print("City-wise revenue:")
city_revenue.show()

City-wise revenue:
+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Hyderabad|       120000|
|  Chennai|        30000|
+---------+-------------+



In [0]:
# Revenue per dealer

dealer_df = sales.join(sales_dealer, "sale_id").join(dealers.alias("d"), "dealer_id").join(customers.alias("c"), "customer_id").join(cars, "car_id")
dealer_df = dealer_df.withColumn("revenue", col("price") * col("quantity"))

dealer_revenue = dealer_df.groupBy("dealer_id", col("d.name").alias("dealer_name")).agg(sum("revenue").alias("total_revenue"))

print("Revenue per dealer:")
dealer_revenue.show()

Revenue per dealer:
+---------+------------+-------------+
|dealer_id| dealer_name|total_revenue|
+---------+------------+-------------+
|      201|Prime Motors|        75000|
|      202|   City Auto|        30000|
|      203|  Speed Cars|        45000|
+---------+------------+-------------+



In [0]:
# Top dealers by revenue

top_dealers = dealer_revenue.orderBy(desc("total_revenue"))

print("Top dealers by revenue:")
top_dealers.show()

Top dealers by revenue:
+---------+------------+-------------+
|dealer_id| dealer_name|total_revenue|
+---------+------------+-------------+
|      201|Prime Motors|        75000|
|      203|  Speed Cars|        45000|
|      202|   City Auto|        30000|
+---------+------------+-------------+



In [0]:
# Dealer-city contribution

dealer_city = dealer_df.groupBy(col("d.city").alias("dealer_city"), "dealer_id").agg(sum("revenue").alias("dealer_city_revenue"))

print("Dealer-city contribution:")
dealer_city.show()

Dealer-city contribution:
+-----------+---------+-------------------+
|dealer_city|dealer_id|dealer_city_revenue|
+-----------+---------+-------------------+
|  Hyderabad|      201|              75000|
|    Chennai|      202|              30000|
|  Bangalore|      203|              45000|
+-----------+---------+-------------------+



In [0]:
# Top 3 customers per city using SQL

df.createOrReplaceTempView("sales_data")

top_customers_per_city = spark.sql("""
    SELECT customer_id, name, city, total_revenue
    FROM (
        SELECT customer_id, name, city, SUM(revenue) as total_revenue,
               ROW_NUMBER() OVER (PARTITION BY city ORDER BY SUM(revenue) DESC) as rank
        FROM sales_data
        GROUP BY customer_id, name, city
    )
    WHERE rank <= 3
    ORDER BY city, rank
""")

print("Top 3 customers per city:")
top_customers_per_city.show()

Top 3 customers per city:
+-----------+-------+---------+-------------+
|customer_id|   name|     city|total_revenue|
+-----------+-------+---------+-------------+
|          5|Charlie|  Chennai|        30000|
|          2|  Alice|  Chennai|            0|
|          1|   John|Hyderabad|        75000|
|          4|    Bob|Hyderabad|        45000|
+-----------+-------+---------+-------------+



In [0]:
# Monthly trends using SQL

monthly_trends = spark.sql("""
    SELECT 
        DATE_FORMAT(sale_date, 'yyyy-MM') as month,
        COUNT(DISTINCT customer_id) as unique_customers,
        SUM(quantity) as total_quantity,
        SUM(revenue) as total_revenue
    FROM sales_data
    GROUP BY DATE_FORMAT(sale_date, 'yyyy-MM')
    ORDER BY month
""")

print("Monthly trends:")
monthly_trends.show()

Monthly trends:
+-------+----------------+--------------+-------------+
|  month|unique_customers|total_quantity|total_revenue|
+-------+----------------+--------------+-------------+
|2024-01|               2|             3|        30000|
|2024-02|               1|             1|        45000|
|2024-03|               2|             4|        75000|
+-------+----------------+--------------+-------------+



In [0]:
# Repeat customers using SQL

repeat_customers = spark.sql("""
    SELECT customer_id, name, COUNT(DISTINCT sale_id) as purchase_count
    FROM sales_data
    GROUP BY customer_id, name
    HAVING COUNT(DISTINCT sale_id) > 1
    ORDER BY purchase_count DESC
""")

print("Repeat customers:")
repeat_customers.show()

Repeat customers:
+-----------+----+--------------+
|customer_id|name|purchase_count|
+-----------+----+--------------+
|          1|John|             2|
+-----------+----+--------------+



In [0]:
# Save final outputs

output_path = "/tmp/car_sales_output/"

customer_revenue.write.mode("overwrite").parquet(f"{output_path}customer_revenue")
brand_sales.write.mode("overwrite").parquet(f"{output_path}brand_sales")
city_revenue.write.mode("overwrite").parquet(f"{output_path}city_revenue")
dealer_revenue.write.mode("overwrite").parquet(f"{output_path}dealer_revenue")
top_customers_per_city.write.mode("overwrite").parquet(f"{output_path}top_customers_per_city")
monthly_trends.write.mode("overwrite").parquet(f"{output_path}monthly_trends")
repeat_customers.write.mode("overwrite").parquet(f"{output_path}repeat_customers")

print("All outputs saved successfully")